In [ ]:
import copy

import pathlib

from laserfarm import GeotiffWriter, MacroPipeline

# Macro-Pipeline AHN Workflow - GeoTIFF Export (All Points)

## Set input/output paths

In [ ]:
root_path = pathlib.Path("/project/lidarac/Share/users/student[ID]")

# input path (targets)
input_path = root_path / "targets_all"

# output path (rasterized targets)
output_path = root_path / "geotiff_all"

In [ ]:
feature_dirs = [
    el for el in input_path.iterdir() if not el.match("tile_*_*.log")
]
print(f"Extract geotiffs for: {len(feature_dirs)} features")

## Setup Cluster

Setup Dask cluster used for the macro-pipeline calculation.

In [ ]:
from dask.distributed import Client

client = Client("tcp://10.0.0.52:33961")
client

## GeoTIFF Export

Export the rasterized features from the target grid to GeoTIFF files.

In [ ]:
# output handle: AHN dataset, target grid spacing 10m, all points
output_handle = "AHN_10m_all_"

# setup input dictionary to configure the geotiff export pipeline
geotiff_export_input_all = {
    "parse_point_cloud": {},
    "data_split": {
        "xSub": 1,
        "ySub": 1,
    },
    "create_subregion_geotiffs": {
        "output_handle": output_handle,
    },
}

In [ ]:
macro = MacroPipeline()

In [ ]:
for feature_dir in feature_dirs:
    gw = GeotiffWriter(bands=feature_dir.name, label=feature_dir.name)
    geotiff_export_input_all_ = copy.deepcopy(geotiff_export_input_all)
    geotiff_export_input_all_["setup_local_fs"] = {
        "input_folder": feature_dir.as_posix(),
        "output_folder": output_path.as_posix(),
    }
    gw.config(geotiff_export_input_all_)
    macro.add_task(gw)

In [ ]:
macro.setup_cluster(cluster=client.scheduler.address)

In [ ]:
# run!
macro.run()

In [ ]:
# save outcome results
macro.print_outcome(to_file="geotiff_export_all.out")

In [ ]:
assert not macro.get_failed_pipelines(), "Some of the tasks have failed!"

## Terminate cluster

In [ ]:
# client.close()
# macro.shutdown()